# Meshes

**Geometry type:** `mesh`

Triangulated 3-D surface meshes. Cell boundaries from EM segmentation,
brain surfaces from MRI, organelle hulls.

1. Generate a synthetic closed surface (sphere approximation)
2. Write with per-vertex attributes (normals, curvature)
3. Open lazily and inspect
4. Read all geometry
5. Spatial bounding-box query
6. Multi-mesh store (cell segmentation)
7. OBJ round-trip (ingest + export)
8. Validate


In [ ]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import ZarrVectorStore
from zarr_vectors.types.meshes import write_mesh, read_mesh
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE       = os.path.join(_tmpdir, "brain_surface.zarrvectors")
STORE_MULTI = os.path.join(_tmpdir, "cells.zarrvectors")
OBJ_PATH    = os.path.join(_tmpdir, "surface.obj")
OBJ_OUT     = os.path.join(_tmpdir, "surface_out.obj")
print("Stores:", STORE, STORE_MULTI)


## 1 · Generate a synthetic closed surface

In [ ]:
def uv_sphere(radius, lat_steps=40, lon_steps=80, offset=(0., 0., 0.)):
    """Generate a UV-sphere with positions, faces, normals, curvature."""
    verts, norms = [], []
    for i in range(lat_steps + 1):
        theta = np.pi * i / lat_steps
        for j in range(lon_steps):
            phi = 2 * np.pi * j / lon_steps
            x = radius * np.sin(theta) * np.cos(phi)
            y = radius * np.sin(theta) * np.sin(phi)
            z = radius * np.cos(theta)
            verts.append([x + offset[0], y + offset[1], z + offset[2]])
            norms.append([np.sin(theta)*np.cos(phi),
                          np.sin(theta)*np.sin(phi),
                          np.cos(theta)])
    verts = np.array(verts, dtype=np.float32)
    norms = np.array(norms, dtype=np.float32)

    faces = []
    for i in range(lat_steps):
        for j in range(lon_steps):
            a = i * lon_steps + j
            b = a + lon_steps
            c = a + 1 if j < lon_steps - 1 else i * lon_steps
            d = b + 1 if j < lon_steps - 1 else (i + 1) * lon_steps
            if i > 0:
                faces.append([a, b, c])
            if i < lat_steps - 1:
                faces.append([b, d, c])
    faces = np.array(faces, dtype=np.int32)

    # Synthetic curvature = 1/radius (constant for sphere, scaled by noise)
    rng_c = np.random.default_rng(42)
    curvature = np.full(len(verts), 1.0 / radius, dtype=np.float32)
    curvature += rng_c.normal(0, 0.005, len(verts)).astype(np.float32)

    return verts, faces, norms, curvature

# Brain-surface-like hemisphere: 150 mm radius centred at (150, 150, 150)
vertices, faces, normals, curvature = uv_sphere(
    radius=50., lat_steps=50, lon_steps=100,
    offset=(150., 150., 150.)
)
print(f"vertices  : {vertices.shape}")
print(f"faces     : {faces.shape}")
print(f"normals   : {normals.shape}")
print(f"curvature : [{curvature.min():.4f}, {curvature.max():.4f}]")


## 2 · Write with per-vertex attributes

In [ ]:
write_mesh(
    STORE,
    vertices=vertices,
    faces=faces,
    chunk_shape=(50.0, 50.0, 50.0),
    bin_shape=(12.5, 12.5, 12.5),
    winding_order="ccw",
    attributes={
        "normal":    normals,    # (N, 3) vector attribute
        "curvature": curvature,  # (N,)   scalar attribute
    },
    coordinate_system="RAS",
    axis_units="millimeter",
)
print("Write complete.")


## 3 · Open lazily and inspect

In [ ]:
store = ZarrVectorStore(STORE)
print(f"geometry_type : {store.geometry_type}")
print(f"spatial_dims  : {store.spatial_dims}")
print(f"n_objects     : {store.n_objects}  (1 mesh surface)")
print(f"vertex_count  : {store.vertex_count(0):,}")
print(f"bounding_box  : lo={store.bounding_box[0].round(1)}"
      f"  hi={store.bounding_box[1].round(1)}")


## 4 · Read all geometry

In [ ]:
result = read_mesh(STORE)
print(f"vertex_count : {result['vertex_count']:,}")
print(f"face_count   : {result['face_count']:,}")
print(f"vertices     : {result['vertices'].shape}")
print(f"faces        : {result['faces'].shape}   (global vertex indices)")
print(f"normal       : {result['attributes']['normal'].shape}")
print(f"curvature    : {result['attributes']['curvature'].shape}")

# Verify normals are unit vectors
norms_len = np.linalg.norm(result['attributes']['normal'], axis=1)
print(f"\nNormal lengths: min={norms_len.min():.4f}  max={norms_len.max():.4f}  (should be ~1.0)")

# Verify face winding (spot-check a few faces)
v = result['vertices']
f = result['faces']
# Cross product should point outward (away from sphere centre ~(150,150,150))
centre = np.array([150., 150., 150.])
normals_from_faces = []
for face_row in f[:5]:
    a, b, c = v[face_row[0]], v[face_row[1]], v[face_row[2]]
    n = np.cross(b - a, c - a)
    centroid = (a + b + c) / 3
    normals_from_faces.append(np.dot(n, centroid - centre))
print(f"Face normal dot (outward): {[f'{x:.1f}' for x in normals_from_faces]}")
print("  (positive = outward-facing ✓)")


## 5 · Spatial bounding-box query

In [ ]:
lo = np.array([100., 100., 100.])
hi = np.array([200., 200., 200.])

bbox_result = read_mesh(STORE, bbox=(lo, hi))
print(f"Faces (centroid in bbox) : {bbox_result['face_count']:,}")
print(f"Vertices in result       : {bbox_result['vertex_count']:,}")
print("  (vertices may extend outside bbox — face centroid is the assignment criterion)")


## 6 · Multi-mesh store (cell segmentation)

In [ ]:
# Generate 20 small sphere-shaped cells at random positions
rng = np.random.default_rng(5)
n_cells = 20
cell_verts_list = []
cell_faces_list = []
cell_volumes    = []
cell_types      = rng.integers(0, 3, n_cells).astype(np.int32)

for i in range(n_cells):
    radius = rng.uniform(8, 20)
    cx, cy, cz = rng.uniform(50, 950, 3)
    v, f, _, _ = uv_sphere(radius, lat_steps=15, lon_steps=30,
                             offset=(cx, cy, cz))
    cell_verts_list.append(v)
    cell_faces_list.append(f)
    # Approx volume of sphere
    cell_volumes.append(float((4/3) * np.pi * radius**3))

write_mesh(
    STORE_MULTI,
    vertices=cell_verts_list,   # list of per-object vertex arrays
    faces=cell_faces_list,      # list of per-object face arrays
    chunk_shape=(100., 100., 100.),
    bin_shape=(25., 25., 25.),
    object_attributes={
        "volume":    np.array(cell_volumes, dtype=np.float32),
        "cell_type": cell_types,
    },
)
store_m = ZarrVectorStore(STORE_MULTI)
print(f"Multi-mesh store: {store_m.n_objects} cells, "
      f"{store_m.vertex_count(0):,} total vertices")


## 6b · Read individual cells

In [ ]:
# Read cell 3 by object ID
r3 = read_mesh(STORE_MULTI, object_ids=[3])
print(f"Cell 3: {r3['vertex_count']} vertices, {r3['face_count']} faces")

# Spatial query in a region
lo_q, hi_q = np.array([200.,200.,200.]), np.array([600.,600.,600.])
bbox_m = read_mesh(STORE_MULTI, bbox=(lo_q, hi_q), return_object_ids=True)
print(f"Cells with faces in region: {bbox_m['object_ids']}")
print(f"Total faces returned: {bbox_m['face_count']:,}")


## 7 · OBJ round-trip

In [ ]:
# Write a small OBJ from the sphere vertices/faces
with open(OBJ_PATH, "w") as f:
    f.write("# zarr-vectors example sphere\n")
    for v in vertices[:200]:      # first 200 vertices for brevity
        f.write(f"v {v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n")
    for face in faces[:100]:
        # OBJ uses 1-based indices; skip faces that reference vertices > 200
        if face.max() < 200:
            f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")
print(f"Wrote test OBJ: {OBJ_PATH}")

# Ingest OBJ → ZVF
STORE_OBJ = os.path.join(_tmpdir, "from_obj.zarrvectors")
from zarr_vectors.ingest.obj import ingest_obj
ingest_obj(OBJ_PATH, STORE_OBJ, chunk_shape=(50., 50., 50.))
r_obj = read_mesh(STORE_OBJ)
print(f"Ingested OBJ: {r_obj['vertex_count']} vertices, {r_obj['face_count']} faces")

# Export back to OBJ
from zarr_vectors.export.obj import export_obj
export_obj(STORE_OBJ, OBJ_OUT)
print(f"Exported to {OBJ_OUT}")


## 8 · Validate

In [ ]:
rv = validate(STORE, level=3)
print(rv.summary())

# For the multi-mesh store, set closed_surface to enable watertightness check
from zarr_vectors.core.store import open_store
root = open_store(STORE_MULTI, mode="r+")
root.attrs["closed_surface"] = True

rv_m = validate(STORE_MULTI, level=4)   # level 4 checks watertightness
print(rv_m.summary())


## Summary

| Step | API |
|------|-----|
| Write single | `write_mesh(path, vertices, faces, chunk_shape, bin_shape, attributes)` |
| Write multi | `write_mesh(path, [verts_list], [faces_list], object_attributes)` |
| Read | `read_mesh(path)` |
| Read by ID | `read_mesh(path, object_ids=[k])` |
| Bbox query | `read_mesh(path, bbox=(lo, hi))` |
| OBJ ingest | `ingest_obj(obj_path, store_path)` |
| OBJ export | `export_obj(store_path, obj_path)` |
